The objective of this is to evaluate RevAI


## Part 0: Setup

Install dependencies and set API keys. **Do not modify this section.**

In [2]:
# Uncomment if running in Colab
# !pip install -qU ragas==0.2.10
# !pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/judithmayzubieta/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/judithmayzubieta/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

In [3]:
from uuid import uuid4

# Each student gets a unique project name
os.environ["LANGCHAIN_PROJECT"] = f"REVAI-SDG-{uuid4().hex[0:8]}"
print(f"LangSmith Project: {os.environ['LANGCHAIN_PROJECT']}")

LangSmith Project: REVAI-SDG-343f7cbf


In [4]:
# os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
os.environ["OPENAI_BASE_URL"] = os.environ.get("OPENAI_BASE_URL") or "https://openai.vocareum.com/v1"
os.environ["OPENAI_API_BASE"] = os.environ["OPENAI_BASE_URL"]

## Part 1: Synthetic Data Generation (Provided – Do Not Modify)

This section generates the evaluation dataset. **Run all cells but do not modify any code.**

> The synthetic dataset must be identical for all experiments so results are comparable.

In [5]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

path = "reviewer/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()
print(f"Loaded {len(docs)} document pages")

Loaded 39 document pages


In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
# generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [26]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)
dataset.to_pandas()

Applying HeadlinesExtractor:   0%|          | 0/27 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/39 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/53 [00:00<?, ?it/s]

Property 'summary' already exists in node 'cce6dc'. Skipping!
Property 'summary' already exists in node '78d73b'. Skipping!
Property 'summary' already exists in node 'b2446f'. Skipping!
Property 'summary' already exists in node 'a66310'. Skipping!
Property 'summary' already exists in node '4fff24'. Skipping!
Property 'summary' already exists in node '27de82'. Skipping!
Property 'summary' already exists in node '85ce7e'. Skipping!
Property 'summary' already exists in node '7d4462'. Skipping!
Property 'summary' already exists in node 'e0400c'. Skipping!
Property 'summary' already exists in node 'cd0ae5'. Skipping!
Property 'summary' already exists in node '9f59e1'. Skipping!
Property 'summary' already exists in node 'ad31d9'. Skipping!
Property 'summary' already exists in node '80e17c'. Skipping!
Property 'summary' already exists in node '500414'. Skipping!
Property 'summary' already exists in node 'e37273'. Skipping!
Property 'summary' already exists in node '52e4b8'. Skipping!
Property

...

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/5 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What are the benefits of using the toolkit pro...,[<1-hop>\n\nTOOLKIT Prompt library: annotated ...,The toolkit prompt library and annotated templ...,multi_hop_abstract_query_synthesizer
1,How do the annotated templates in the toolkit ...,[<1-hop>\n\nTOOLKIT Prompt library: annotated ...,The annotated templates in the toolkit prompt ...,multi_hop_abstract_query_synthesizer
2,What are the annotated templates in the toolki...,[<1-hop>\n\nTOOLKIT Prompt library: annotated ...,The annotated templates in the toolkit prompt ...,multi_hop_abstract_query_synthesizer
3,How can the toolkit prompt library's annotated...,[<1-hop>\n\nTOOLKIT Prompt library: annotated ...,The toolkit prompt library's annotated templat...,multi_hop_abstract_query_synthesizer
4,What are the benefits of using annotated templ...,[<1-hop>\n\nTOOLKIT Prompt library: annotated ...,The benefits of using annotated templates from...,multi_hop_abstract_query_synthesizer


In [28]:
from langsmith import Client

client = Client()
dataset_name = f"RevAI Assignment_1 - {os.environ['LANGCHAIN_PROJECT']}"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="RevAI - Week 14 Graded Assignment_1"
)

for data_row in dataset.to_pandas().iterrows():
    client.create_example(
        inputs={"question": data_row[1]["user_input"]},
        outputs={"answer": data_row[1]["reference"]},
        metadata={"context": data_row[1]["reference_contexts"]},
        dataset_id=langsmith_dataset.id
    )

print(f"Dataset '{dataset_name}' created with {len(dataset.to_pandas())} examples")

Dataset 'RevAI Assignment_1 - REVAI-SDG-343f7cbf' created with 5 examples


## Part 2: Evaluators (Provided – Do Not Modify)

These evaluators remain constant across ALL experiments.

In [29]:
from langchain_openai import ChatOpenAI

# eval_llm = ChatOpenAI(model="gpt-4.1")
eval_llm = ChatOpenAI(model="gpt-4o-mini")

In [30]:

from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm": eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm": eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm": eval_llm
    }
)

evaluators = [qa_evaluator, labeled_helpfulness_evaluator, empathy_evaluator]
print("Evaluators ready: QA Correctness, Labeled Helpfulness, Empathy")

Evaluators ready: QA Correctness, Labeled Helpfulness, Empathy


## Part 3: Baseline Pipeline – Run 1 of 4 (Do Not Modify)

This is the baseline configuration. Run it and record the scores.

| Parameter | Baseline Value |
|-----------|---------------|
| Chunk Size | 1000 |
| Chunk Overlap | 100 |
| Embedding | text-embedding-3-small |
| Retrieval k | 10 |
| LLM | gpt-4.o-mini |
| Prompt | Basic context + question |

In [31]:
# === BASELINE RAG PIPELINE ===
from langchain.text_splitter import RecursiveCharacterTextSplitter

baseline_documents = docs
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
baseline_documents = text_splitter.split_documents(baseline_documents)
print(f"Baseline: {len(baseline_documents)} chunks (chunk_size=1000, overlap=100)")

Baseline: 117 chunks (chunk_size=1000, overlap=100)


In [32]:
from langchain_qdrant import QdrantVectorStore

baseline_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
baseline_vectorstore = QdrantVectorStore.from_documents(
    documents=baseline_documents,
    embedding=baseline_embeddings,
    location=":memory:",
    collection_name="baseline"
 )
baseline_retriever = baseline_vectorstore.as_retriever(search_kwargs={"k": 10})

In [33]:
from langchain.prompts import ChatPromptTemplate

BASELINE_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You are helpful and empathetic. You must answer the question correctly and with empathy based only on context.

Context: {context}
Question: {question}
"""

baseline_prompt = ChatPromptTemplate.from_template(BASELINE_PROMPT)

In [34]:
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

# baseline_llm = ChatOpenAI(model="gpt-4.1-mini")
baseline_llm = ChatOpenAI(model="gpt-4o-mini")

baseline_chain = (
    {"context": itemgetter("question") | baseline_retriever, "question": itemgetter("question")}
    | baseline_prompt | baseline_llm | StrOutputParser()
)

# Quick test
print(baseline_chain.invoke({"question": "What are the key capabilities of LLM in test tasks?"}))

The key capabilities of LLMs in test tasks include:

1. **Requirements analysis and improvement**: Finding ambiguities, inconsistencies, and gaps in the test basis; generating clarification questions for stakeholders.
2. **Test case creation support**: Drafting test cases and suggesting test objectives from requirements or user stories.
3. **Test oracle generation**: Helping produce expected results.
4. **Test data generation**: Creating datasets, boundary values, and combinations.
5. **Test automation support**: Generating scripts from test case descriptions, improving existing scripts, and suggesting techniques.
6. **Test result analysis**: Summarizing results and classifying anomalies by severity and priority.
7. **Testware creation**: Drafting and keeping updated test plans, test reports, and defect reports.

These capabilities enhance the efficiency and effectiveness of the testing process, supporting testers in various aspects of their work.


### ⚠️ Baseline Evaluation – Run 1 of 4

In [35]:
baseline_results = evaluate(
    baseline_chain.invoke,
    data=dataset_name,
    evaluators=evaluators,
    metadata={"revision_id": "baseline"},
)
print("\n=== BASELINE RESULTS ===")
print(baseline_results)

View the evaluation results for experiment: 'enchanted-football-11' at:
https://smith.langchain.com/o/49a02b9f-6a05-443d-9275-6e69d74840d8/datasets/cbb571a7-68a7-40da-8e00-cc3d59f42afc/compare?selectedSessions=358b8a1d-412f-49ec-8028-e5bd3d9d0315




0it [00:00, ?it/s]


=== BASELINE RESULTS ===
<ExperimentResults enchanted-football-11>


### Experiment 1: Chunk size set to 500 – Run 2 of 4

**Variable Changed:** Chunk Size

**Hypothesis:** I expect that the correctness and helpfulness drop because smaller chunk size provides smaller context or data, though it may give a more targeted response.

In [ ]:
# # ============================================================
# # EXPERIMENT 1: Change the chunk size for document splitting
# # Variable Changed: chunk_size = 500
# # ============================================================

# # TODO: Build your modified pipeline below.
# # Only change ONE variable from the baseline.

# # --- Your code here ---
# from langchain.text_splitter import RecursiveCharacterTextSplitter

# baseline_documents = docs
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
# baseline_documents = text_splitter.split_documents(baseline_documents)
# print(f"Experiment 1: {len(baseline_documents)} chunks (chunk_size=500, overlap=50)")



# # --- Build the chain ---
# experiment_1_chain = (
#     {"context": itemgetter("question") | baseline_retriever, "question": itemgetter("question")}
#     | baseline_prompt | baseline_llm | StrOutputParser()
# )


Experiment 1: 216 chunks (chunk_size=500, overlap=50)


In [ ]:
# # --- Evaluate Experiment 1 (Run 2 of 4) ---
# experiment_1_results = evaluate(
#     experiment_1_chain.invoke,
#     data=dataset_name,
#     evaluators=evaluators,
#     metadata={"revision_id": "Updated prompt_chunk_size_500_overlap_50"},
# )
# print("\n=== EXPERIMENT 1 RESULTS ===")
# print(experiment_1_results)


View the evaluation results for experiment: 'flowery-position-59' at:
https://smith.langchain.com/o/49a02b9f-6a05-443d-9275-6e69d74840d8/datasets/4a73b01c-3f88-4aed-8098-4d35952bb535/compare?selectedSessions=0640a9ed-ad47-468f-81eb-7b93d9dc6219




0it [00:00, ?it/s]


=== EXPERIMENT 1 RESULTS ===
<ExperimentResults flowery-position-59>


---

### Experiment 2: Retrieval k = 3 – Run 3 of 4

**Variable Changed:** Retrieval k

**Hypothesis:** I expect that the correctness will drop because decreasing the value of k retrieves less context.

In [ ]:
# # ============================================================
# # EXPERIMENT 2: Decreased the number of retrieved documents from 10 to 3
# # Variable Changed: Retrieval parameter k = 3
# # ============================================================

# # TODO: Build your modified pipeline below.
# # Only change ONE variable from the baseline.

# # --- Your code here ---
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain_qdrant import QdrantVectorStore

# #Setting to the original/baseline chunk size
# baseline_documents = docs
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
# baseline_documents = text_splitter.split_documents(baseline_documents)
# print(f"Experiment 2: {len(baseline_documents)} chunks (chunk_size=1000, overlap=100)")

# baseline_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# baseline_vectorstore = QdrantVectorStore.from_documents(
#     documents=baseline_documents,
#     embedding=baseline_embeddings,
#     location=":memory:",
#     collection_name="baseline"
#  )
# #Update the retriever to use k=3 instead of the baseline k=10
# baseline_retriever_updated = baseline_vectorstore.as_retriever(search_kwargs={"k": 3})
# print(f"Experiment 2: Retrieval parameter k = 3 (decreased from 10)")


# experiment_2_chain = (
#     {"context": itemgetter("question") | baseline_retriever_updated, "question": itemgetter("question")}
#     | baseline_prompt | baseline_llm | StrOutputParser()
# )


Experiment 2: 117 chunks (chunk_size=1000, overlap=100)
Experiment 2: Retrieval parameter k = 3 (decreased from 10)


In [ ]:
# # --- Evaluate Experiment 2 (Run 3 of 4) ---
# experiment_2_results = evaluate(
#     experiment_2_chain.invoke,
#     data=dataset_name,
#     evaluators=evaluators,
#     metadata={"revision_id": "chunksize_1000_RETRIEVAL_k_3"},
# )
# print("\n=== EXPERIMENT 2 RESULTS ===")
# print(experiment_2_results)


View the evaluation results for experiment: 'perfect-country-14' at:
https://smith.langchain.com/o/49a02b9f-6a05-443d-9275-6e69d74840d8/datasets/4a73b01c-3f88-4aed-8098-4d35952bb535/compare?selectedSessions=ebeb68f7-2d52-4880-ad10-d172bc3b7ac9




0it [00:00, ?it/s]


=== EXPERIMENT 2 RESULTS ===
<ExperimentResults perfect-country-14>


---

### Experiment 3: Update the embedding model to text-embedding-3-large – Run 4 of 4

**Variable Changed:** Prompt

**Hypothesis:** I expect that empathy increases because a persona can boost the tone and validation.

In [ ]:
# ============================================================
# EXPERIMENT 3: Updated to text-embedding-3-large
# Variable Changed: model - updated to text-embedding-3-large 
# ============================================================

# TODO: Build your modified pipeline below.
# Only change ONE variable from the baseline.

# --- Your code here ---
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore

#Setting to the original/baseline chunk size
baseline_documents = docs
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
baseline_documents = text_splitter.split_documents(baseline_documents)
print(f"Experiment 3: {len(baseline_documents)} chunks (chunk_size=1000, overlap=100)")

baseline_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
baseline_vectorstore = QdrantVectorStore.from_documents(
    documents=baseline_documents,
    embedding=baseline_embeddings,
    location=":memory:",
    collection_name="baseline"
 )

#Setting retriever back to k=10 from the baseline value
baseline_retriever = baseline_vectorstore.as_retriever(search_kwargs={"k": 10})
print(f"Experiment 3: Retrieval parameter k = 10 (baseline)")

#Update the prompt to include a role for the model
from langchain.prompts import ChatPromptTemplate

baseline_llm = ChatOpenAI(model="gpt-4o-mini")

UPDATED_PROMPT = """\
Given a provided context and question, you must answer the question with empathy and based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You are a helpful and empathetic assistant that provides clear and concise answers to questions related to the provided context. Your responses should be informative, accurate, and considerate of the user's perspective.

Context: {context}
Question: {question}
"""

updated_prompt = ChatPromptTemplate.from_template(UPDATED_PROMPT)

# --- Build the chain ---
experiment_3_chain = (
    {"context": itemgetter("question") | baseline_retriever, "question": itemgetter("question")}
    | updated_prompt | baseline_llm | StrOutputParser()
)


Experiment 3: 117 chunks (chunk_size=1000, overlap=100)
Experiment 3: Retrieval parameter k = 10 (baseline)


In [37]:
# --- Evaluate Experiment 3 (Run 4 of 4) ---
experiment_3_results = evaluate(
    experiment_3_chain.invoke,
    data=dataset_name,
    evaluators=evaluators,
    metadata={"revision_id": "embedding model is large_UPDATED_PROMPT"},
)
print("\n=== EXPERIMENT 3 RESULTS ===")
print(experiment_3_results)


View the evaluation results for experiment: 'new-hand-42' at:
https://smith.langchain.com/o/49a02b9f-6a05-443d-9275-6e69d74840d8/datasets/cbb571a7-68a7-40da-8e00-cc3d59f42afc/compare?selectedSessions=d6657014-3ab0-497a-858b-7cf159a549a8




0it [00:00, ?it/s]


=== EXPERIMENT 3 RESULTS ===
<ExperimentResults new-hand-42>
